<a href="https://colab.research.google.com/github/x-shakuni/machine-learning-projects/blob/main/ensemble_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import StackingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
import seaborn as sns

In [ ]:
df = sns.load_dataset('iris')

In [ ]:
df.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [ ]:
X = df.drop('species', axis=1)
y = df['species']

In [ ]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

## STACKING

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify = y_encoded)

In [ ]:
base_learners = [
    ('dt', DecisionTreeClassifier(random_state=42)),
    ('svm', SVC(probability=True, kernel='rbf', random_state=42)),
    ('lr', LogisticRegression(max_iter=1000))
]

In [ ]:
meta_learner = LogisticRegression(max_iter=1000)

In [ ]:
stacking_clf = StackingClassifier(
    estimators=base_learners,
    final_estimator=meta_learner,
    cv=5
    )

In [ ]:
stacking_clf.fit(X_train, y_train)

StackingClassifier(cv=5,
                   estimators=[('dt', DecisionTreeClassifier(random_state=42)),
                               ('svm', SVC(probability=True, random_state=42)),
                               ('lr', LogisticRegression(max_iter=1000))],
                   final_estimator=LogisticRegression(max_iter=1000))

In [ ]:
y_pred = stacking_clf.predict(X_test)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")

Accuracy: 0.9666666666666667


# BAGGING


In [ ]:
## random forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
model_rf = RandomForestClassifier(
    n_estimators=100,     # number of trees
    max_depth=None,       # let trees grow fully
    random_state=42
    )

In [ ]:
model_rf.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [ ]:
y_pred = model_rf.predict(X_test)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")

Accuracy: 0.9


# BOOSTING

Boosting is an ensemble learning technique that converts multiple weak learners (models slightly better than random chance) into a single strong learner by training them sequentially, with each model correcting the errors of its predecessor.


How Boosting Works
* The core mechanism follows these steps:

* Initialize weights — assign equal weights to all training samples

* Train a weak learner — fit a simple model (usually a shallow decision tree/stump)

* Identify misclassifications — find samples the model got wrong

* Reweight samples — increase weights on misclassified points so the next model focuses on them

* Repeat sequentially — each new learner targets the previous model's weaknesses

* Aggregate predictions — combine all learners via a weighted sum, where more accurate learners get more say





In [ ]:
from sklearn.ensemble import GradientBoostingClassifier, AdaBoostClassifier

In [ ]:
from xgboost import XGBClassifier

### adaboost

In [ ]:
ada_model = AdaBoostClassifier(n_estimators=100, random_state=42)

In [ ]:
ada_model.fit(X_train, y_train)

AdaBoostClassifier(n_estimators=100, random_state=42)

In [ ]:
y_pred = ada_model.predict(X_test)

In [ ]:
accuracy_ada = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy_ada}")

Accuracy: 0.9333333333333333


### gradientboostingclassifier

In [ ]:
gb_model = GradientBoostingClassifier(n_estimators=100, random_state=42, learning_rate=0.1)

In [ ]:
gb_model.fit(X_train, y_train)

GradientBoostingClassifier(random_state=42)

In [ ]:
y_pred = gb_model.predict(X_test)

In [ ]:
accuracy_gb = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy_gb}")

Accuracy: 0.9666666666666667


### xgboost

In [ ]:
xg_model = XGBClassifier(n_estimators=100, random_state=42, learning_rate=0.1, max_depth=3, use_label_encoder=False, eval_metric='mlogloss')

In [ ]:
xg_model.fit(X_train, le.transform(y_train))

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [15:59:53] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)

In [ ]:
y_pred = xg_model.predict(X_test)

In [ ]:
accuracy_xg = accuracy_score(le.transform(y_test), y_pred)
print(f"Accuracy: {accuracy_xg}")

Accuracy: 0.9333333333333333
